<a href="https://colab.research.google.com/github/abiodunadesesan/FlyRank-ml-Internship-/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abiodunadesesan/FlyRank-ml-Internship-/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

### Research Question

Which visible pages have the greatest opportunity for CTR improvement, and how can we rank them for content optimization?

### Decision

Which webpages should be prioritized for CTR optimization?

### Who will use the result?

Content and SEO teams can use the ranking to decide which pages to review first for possible improvements to titles, meta descriptions, content, and other on-page elements.

### Objective

Build a repeatable CTR opportunity scoring approach using the FlyRank search dataset, compare it with a transparent baseline, validate the result honestly, and turn the analysis into a ranked action playbook.

In [ ]:
import duckdb
import pandas as pd

print("DuckDB:", duckdb.__version__)
print("Pandas:", pd.__version__)

DuckDB: 1.3.2
Pandas: 2.2.3


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("FlyRank warehouse authentication is ready.")

FlyRank warehouse authentication is ready.


In [ ]:
# Inspect the available FlyRank warehouse tables

tables = con.sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name
""").df()

tables

,table_name


In [ ]:
# Check whether the Hugging Face secret is available
print("Token available:", bool(HF_TOKEN))

Token available: True


In [ ]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("FlyRank warehouse authentication is ready.")

FlyRank warehouse authentication is ready.


In [ ]:
rel = "hf://datasets/FlyRank/internship-warehouse"

test = con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

test

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows
0,78835655


In [ ]:
schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [ ]:
summary = con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents,
    COUNT(DISTINCT CONCAT(
        CAST(report_date AS VARCHAR), '|',
        client_hash_id, '|',
        content_hash_id
    )) AS unique_grain_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows,clients,contents,unique_grain_rows
0,2025-01-27,2026-06-30,78835655,70,427292,78829265


In [ ]:
duplicates = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 20
""").df()

duplicates

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-17,client_1a730cb2640a1abf,content_965b9031838c130f,2
1,2026-06-17,client_1a730cb2640a1abf,content_5c5b96cd4d1829d8,2
2,2026-06-18,client_def0955f7a377868,content_40c868cfcb49537c,2
3,2026-06-20,client_1a730cb2640a1abf,content_6af788521680129d,2
4,2026-06-19,client_06d356715a8ff3b6,content_6c6a2658f025ec02,2
5,2026-06-21,client_1a8bf67cad4ee525,content_567e7aa5ce2a36e6,2
6,2026-06-22,client_8ddc46da5414ffd8,content_55335cfcf3499724,2
7,2026-06-23,client_1a8bf67cad4ee525,content_e85eff1aad797f4c,2
8,2026-06-23,client_1a8bf67cad4ee525,content_2630830d5f397c6c,2
9,2026-06-23,client_1a8bf67cad4ee525,content_fb1b290d16ca0d29,2


In [ ]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("FlyRank warehouse connection restored.")

FlyRank warehouse connection restored.


In [ ]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)

Pandas: 2.2.3


In [ ]:
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata

# Restore authentication
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

# Restore DuckDB connection
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

# Restore warehouse location
rel = "hf://datasets/FlyRank/internship-warehouse"

# Restore time windows
TRAIN_START = "2025-10-01"
TRAIN_END   = "2026-03-31"

VALID_START = "2026-04-01"
VALID_END   = "2026-06-30"

print("Capstone state restored.")
print("Training:", TRAIN_START, "to", TRAIN_END)
print("Validation:", VALID_START, "to", VALID_END)

Capstone state restored.
Training: 2025-10-01 to 2026-03-31
Validation: 2026-04-01 to 2026-06-30


In [ ]:
page_features = con.sql(f"""
WITH clean_data AS (
    SELECT DISTINCT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE gsc_data_available = TRUE
      AND gsc_impressions > 0
),

page_features AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,

        SUM(gsc_impressions * gsc_avg_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS avg_position,

        COUNT(DISTINCT report_date) AS active_days

    FROM clean_data

    WHERE report_date BETWEEN '{TRAIN_START}' AND '{TRAIN_END}'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM page_features
WHERE impressions >= 100
""").df()

print("Training pages:", len(page_features))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training pages: 126622


In [ ]:
validation_outcomes = con.sql(f"""
WITH clean_data AS (
    SELECT DISTINCT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE gsc_data_available = TRUE
      AND gsc_impressions > 0
),

future_pages AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS validation_impressions,
        SUM(gsc_clicks) AS validation_clicks,

        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS validation_ctr,

        SUM(gsc_impressions * gsc_avg_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS validation_avg_position,

        COUNT(DISTINCT report_date) AS validation_active_days

    FROM clean_data

    WHERE report_date BETWEEN '{VALID_START}' AND '{VALID_END}'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM future_pages
WHERE validation_impressions >= 100
""").df()

print("Validation pages:", len(validation_outcomes))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Validation pages: 151996


In [ ]:
model_dataset = page_features.merge(
    validation_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Pages in both periods:", len(model_dataset))

Pages in both periods: 104173


In [ ]:
model_dataset["position_bucket"] = pd.cut(
    model_dataset["avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "4-5", "6-10", "11-20", "21+"],
    right=True
)

print(model_dataset["position_bucket"].value_counts().sort_index())

position_bucket
1-3      10519
4-5      17472
6-10     33221
11-20    21735
21+      21226
Name: count, dtype: int64


In [ ]:
baseline_ctr = {
    "1-3": 0.005443,
    "4-5": 0.004597,
    "6-10": 0.003267,
    "11-20": 0.003472,
    "21+": 0.001559
}

model_dataset["baseline_ctr"] = (
    model_dataset["position_bucket"]
    .astype(str)
    .map(baseline_ctr)
)

print(
    "Pages with baseline CTR:",
    model_dataset["baseline_ctr"].notna().sum()
)

model_dataset[
    ["position_bucket", "ctr", "baseline_ctr"]
].head(10)

Pages with baseline CTR: 104173


,position_bucket,ctr,baseline_ctr
0,21+,0.001198,0.001559
1,21+,0.005769,0.001559
2,21+,0.007663,0.001559
3,21+,0.002451,0.001559
4,11-20,0.005325,0.003472
5,4-5,0.001078,0.004597
6,21+,0.002463,0.001559
7,6-10,0.000663,0.003267
8,21+,0.009412,0.001559
9,21+,0.003914,0.001559


In [8]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("FlyRank warehouse connection restored.")

FlyRank warehouse connection restored.


In [9]:
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata

# -----------------------------
# 1. Restore connection
# -----------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

TRAIN_START = "2025-10-01"
TRAIN_END   = "2026-03-31"

VALID_START = "2026-04-01"
VALID_END   = "2026-06-30"


# -----------------------------
# 2. Build training features
# -----------------------------

page_features = con.sql(f"""
WITH clean_data AS (
    SELECT DISTINCT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE gsc_data_available = TRUE
      AND gsc_impressions > 0
)

SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,

    SUM(gsc_clicks) * 1.0
        / NULLIF(SUM(gsc_impressions), 0) AS ctr,

    SUM(gsc_impressions * gsc_avg_position) * 1.0
        / NULLIF(SUM(gsc_impressions), 0) AS avg_position,

    COUNT(DISTINCT report_date) AS active_days

FROM clean_data

WHERE report_date BETWEEN '{TRAIN_START}' AND '{TRAIN_END}'

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(gsc_impressions) >= 100
""").df()


# -----------------------------
# 3. Build future outcomes
# -----------------------------

validation_outcomes = con.sql(f"""
WITH clean_data AS (
    SELECT DISTINCT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE gsc_data_available = TRUE
      AND gsc_impressions > 0
)

SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS validation_impressions,
    SUM(gsc_clicks) AS validation_clicks,

    SUM(gsc_clicks) * 1.0
        / NULLIF(SUM(gsc_impressions), 0) AS validation_ctr,

    SUM(gsc_impressions * gsc_avg_position) * 1.0
        / NULLIF(SUM(gsc_impressions), 0) AS validation_avg_position,

    COUNT(DISTINCT report_date) AS validation_active_days

FROM clean_data

WHERE report_date BETWEEN '{VALID_START}' AND '{VALID_END}'

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(gsc_impressions) >= 100
""").df()


# -----------------------------
# 4. Join historical + future
# -----------------------------

model_dataset = page_features.merge(
    validation_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)


# -----------------------------
# 5. Position buckets
# -----------------------------

model_dataset["position_bucket"] = pd.cut(
    model_dataset["avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "4-5", "6-10", "11-20", "21+"],
    right=True
)


# -----------------------------
# 6. Historical CTR baseline
# -----------------------------

baseline_ctr = {
    "1-3": 0.005443,
    "4-5": 0.004597,
    "6-10": 0.003267,
    "11-20": 0.003472,
    "21+": 0.001559
}

model_dataset["baseline_ctr"] = (
    model_dataset["position_bucket"]
    .astype(str)
    .map(baseline_ctr)
)


# -----------------------------
# 7. Future CTR opportunity
# -----------------------------

model_dataset["ctr_gap"] = (
    model_dataset["baseline_ctr"]
    - model_dataset["validation_ctr"]
)

model_dataset["potential_clicks"] = (
    model_dataset["ctr_gap"]
    * model_dataset["validation_impressions"]
)


# -----------------------------
# 8. Verify
# -----------------------------

print("Training pages:", len(page_features))
print("Validation pages:", len(validation_outcomes))
print("Pages in both periods:", len(model_dataset))
print(
    "Pages with positive CTR gap:",
    (model_dataset["ctr_gap"] > 0).sum()
)
print(
    "Pages with zero/negative CTR gap:",
    (model_dataset["ctr_gap"] <= 0).sum()
)

print("\nSample:")
display(
    model_dataset[
        [
            "position_bucket",
            "baseline_ctr",
            "validation_ctr",
            "ctr_gap",
            "validation_impressions",
            "potential_clicks"
        ]
    ].head(10)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training pages: 126622
Validation pages: 151996
Pages in both periods: 104173
Pages with positive CTR gap: 77950
Pages with zero/negative CTR gap: 26223

Sample:


,position_bucket,baseline_ctr,validation_ctr,ctr_gap,validation_impressions,potential_clicks
0,11-20,0.003472,0.000000,0.003472,366.0,1.270752
1,21+,0.001559,0.004104,-0.002545,731.0,-1.860371
2,21+,0.001559,0.002037,-0.000478,1964.0,-0.938124
3,21+,0.001559,0.001241,0.000318,1611.0,0.511549
4,21+,0.001559,0.013436,-0.011877,1042.0,-12.375522
5,21+,0.001559,0.000000,0.001559,427.0,0.665693
6,11-20,0.003472,0.003401,0.000071,294.0,0.020768
7,21+,0.001559,0.002288,-0.000729,1748.0,-1.274868
8,11-20,0.003472,0.009153,-0.005681,437.0,-2.482736
9,6-10,0.003267,0.004213,-0.000946,712.0,-0.673896


In [10]:
print("model_dataset:", len(model_dataset))
print("page_features:", len(page_features))
print("validation_outcomes:", len(validation_outcomes))
print("opportunity_score exists:", "opportunity_score" in model_dataset.columns)

model_dataset: 104173
page_features: 126622
validation_outcomes: 151996
opportunity_score exists: False


In [12]:
import pandas as pd
import numpy as np

# Create historical CTR gap
model_dataset["historical_ctr_gap"] = (
    model_dataset["baseline_ctr"]
    - model_dataset["ctr"]
)

# Create opportunity score
model_dataset["opportunity_score"] = (
    model_dataset["historical_ctr_gap"].clip(lower=0)
    * np.log1p(model_dataset["impressions"])
)

print("Opportunity score created:", "opportunity_score" in model_dataset.columns)
print(model_dataset["opportunity_score"].describe())

Opportunity score created: True
count    104173.000000
mean          0.012219
std           0.011333
min           0.000000
25%           0.000000
50%           0.010094
75%           0.020595
max           0.069787
Name: opportunity_score, dtype: float64


In [13]:
top_opportunities = (
    model_dataset
    .sort_values("opportunity_score", ascending=False)
    [[
        "client_hash_id",
        "content_hash_id",
        "position_bucket",
        "ctr",
        "baseline_ctr",
        "historical_ctr_gap",
        "impressions",
        "active_days",
        "opportunity_score"
    ]]
    .head(20)
)

top_opportunities

,client_hash_id,content_hash_id,position_bucket,ctr,baseline_ctr,historical_ctr_gap,impressions,active_days,opportunity_score
57129,client_73cda7b4e4f265ea,content_fec55986a1868d62,1-3,0.000003,0.005443,0.005440,372381.0,182,0.069787
5107,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1-3,0.000011,0.005443,0.005432,366197.0,182,0.069590
10565,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1-3,0.000007,0.005443,0.005436,279673.0,135,0.068173
87800,client_23a62021009f63c4,content_44f34c0a90047651,1-3,0.000132,0.005443,0.005311,302868.0,65,0.067030
77007,client_e547b89c05043229,content_306bc78dff1eb683,1-3,0.000504,0.005443,0.004939,242280.0,135,0.061239
33084,client_23a62021009f63c4,content_2ac8c7995de53cd1,1-3,0.000191,0.005443,0.005252,94198.0,78,0.060151
77010,client_e547b89c05043229,content_c46df0fa61530d86,1-3,0.000487,0.005443,0.004956,170379.0,135,0.059697
23825,client_e547b89c05043229,content_35f10273f0baf406,1-3,0.000214,0.005443,0.005229,79593.0,135,0.059012
47681,client_23a62021009f63c4,content_bf078007df823490,1-3,0.000000,0.005443,0.005443,44707.0,20,0.058283
8418,client_23a62021009f63c4,content_66d3e7a515e4ec68,1-3,0.000132,0.005443,0.005311,53174.0,171,0.057795


In [ ]:
print("CTR gap statistics")
print(model_dataset["ctr_gap"].describe())

print("\nPotential click statistics")
print(model_dataset["potential_clicks"].describe())

print("\nLargest positive opportunities")
display(
    model_dataset[
        model_dataset["potential_clicks"] > 0
    ][
        [
            "position_bucket",
            "baseline_ctr",
            "validation_ctr",
            "ctr_gap",
            "validation_impressions",
            "potential_clicks"
        ]
    ]
    .sort_values("potential_clicks", ascending=False)
    .head(20)
)

CTR gap statistics
count    104173.000000
mean          0.000941
std           0.003541
min          -0.101081
25%          -0.000017
50%           0.001559
75%           0.003267
max           0.005443
Name: ctr_gap, dtype: float64

Potential click statistics
count    104173.000000
mean          4.184639
std          60.768383
min       -4642.451430
25%          -0.020496
50%           1.138752
75%           4.835160
max        2294.267774
Name: potential_clicks, dtype: float64

Largest positive opportunities


,position_bucket,baseline_ctr,validation_ctr,ctr_gap,validation_impressions,potential_clicks
50422,1-3,0.005443,0.001103,0.004340,528618.0,2294.267774
82796,4-5,0.004597,0.001254,0.003343,611708.0,2045.021676
36167,4-5,0.004597,0.000909,0.003688,508071.0,1873.602387
75116,4-5,0.004597,0.002648,0.001949,908095.0,1769.512715
30861,4-5,0.004597,0.001299,0.003298,523610.0,1727.035170
76902,4-5,0.004597,0.001719,0.002878,596157.0,1715.533729
72581,6-10,0.003267,0.000224,0.003043,557202.0,1695.378934
24828,1-3,0.005443,0.002543,0.002900,565913.0,1641.264459
25300,4-5,0.004597,0.001515,0.003082,504799.0,1555.561003
43921,4-5,0.004597,0.001041,0.003556,413999.0,1472.153403


In [ ]:
# Build a transparent historical CTR opportunity score

# Historical CTR gap
model_dataset["historical_ctr_gap"] = (
    model_dataset["baseline_ctr"] - model_dataset["ctr"]
)

# Only positive gaps represent potential opportunity
model_dataset["positive_ctr_gap"] = (
    model_dataset["historical_ctr_gap"].clip(lower=0)
)

# Log exposure reduces the dominance of extremely high-impression pages
model_dataset["exposure_score"] = np.log1p(
    model_dataset["impressions"]
)

# Stability rewards pages with more observed active days
model_dataset["stability_score"] = (
    model_dataset["active_days"] / 182
).clip(upper=1)

# Transparent action score
model_dataset["opportunity_score"] = (
    model_dataset["positive_ctr_gap"]
    * model_dataset["exposure_score"]
    * model_dataset["stability_score"]
)

print(model_dataset["opportunity_score"].describe())

print("\nTop opportunities based only on historical information:")

display(
    model_dataset[
        [
            "client_hash_id",
            "content_hash_id",
            "position_bucket",
            "ctr",
            "baseline_ctr",
            "historical_ctr_gap",
            "impressions",
            "active_days",
            "opportunity_score"
        ]
    ]
    .sort_values("opportunity_score", ascending=False)
    .head(20)
)

count    104173.000000
mean          0.007450
std           0.009105
min           0.000000
25%           0.000000
50%           0.003826
75%           0.011751
max           0.069787
Name: opportunity_score, dtype: float64

Top opportunities based only on historical information:


,client_hash_id,content_hash_id,position_bucket,ctr,baseline_ctr,historical_ctr_gap,impressions,active_days,opportunity_score
57129,client_73cda7b4e4f265ea,content_fec55986a1868d62,1-3,0.000003,0.005443,0.005440,372381.0,182,0.069787
5107,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1-3,0.000011,0.005443,0.005432,366197.0,182,0.069590
15522,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,1-3,0.000095,0.005443,0.005348,31425.0,182,0.055376
57128,client_73cda7b4e4f265ea,content_c9f840183215651b,4-5,0.000000,0.004597,0.004597,151750.0,182,0.054842
69606,client_73cda7b4e4f265ea,content_254500f1d708cd6b,1-3,0.000264,0.005443,0.005179,37878.0,182,0.054598
2594,client_23a62021009f63c4,content_dcc8191464a7e5b0,1-3,0.000109,0.005443,0.005334,27433.0,182,0.054507
8418,client_23a62021009f63c4,content_66d3e7a515e4ec68,1-3,0.000132,0.005443,0.005311,53174.0,171,0.054302
10879,client_73cda7b4e4f265ea,content_f57e4e8c0208f271,1-3,0.000405,0.005443,0.005038,46892.0,182,0.054185
10886,client_73cda7b4e4f265ea,content_97d7732bcbfc6931,1-3,0.000603,0.005443,0.004840,68038.0,182,0.053863
65056,client_fef1a8f436438636,content_5cc6fa5852bf25f1,1-3,0.000274,0.005443,0.005169,32853.0,182,0.053757


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Method

The analysis uses page-level historical search performance aggregated over the training period from 2025-10-01 to 2026-03-31.

The unit of analysis is a client-content page pair.

Historical features include impressions, clicks, CTR, average position, active days, and position bucket.

The position bucket provides a transparent baseline CTR based on observed CTR within each position range.

### Label

The validation period is 2026-04-01 to 2026-06-30.

The main outcome is the CTR gap:

CTR gap = baseline CTR - validation CTR

A positive CTR gap means the page achieved a lower CTR during validation than the position-based benchmark.

### Baseline

The baseline uses observed CTR by position bucket:

- 1-3: 0.5443%
- 4-5: 0.4597%
- 6-10: 0.3267%
- 11-20: 0.3472%
- 21+: 0.1559%

### Validation design

A time-aware split is used.

Training:
2025-10-01 to 2026-03-31

Validation:
2026-04-01 to 2026-06-30

The opportunity score is calculated using historical information only. Validation-period outcomes are held out until evaluation.

### Leakage check

Future validation CTR, clicks, impressions, position, and other validation outcomes are not used to construct the historical opportunity score.

The validation period is used only to evaluate whether pages ranked as opportunities subsequently show a positive CTR gap.

### Interpretation

The results are directional and intended for decision support. They identify pages that may deserve human review for CTR improvement. They do not establish causal effects or prove how Google's ranking system works.

In [14]:
# Restore capstone period definitions

TRAIN_START = "2025-10-01"
TRAIN_END = "2026-03-31"

VALID_START = "2026-04-01"
VALID_END = "2026-06-30"

print("Training:", TRAIN_START, "to", TRAIN_END)
print("Validation:", VALID_START, "to", VALID_END)

print("model_dataset exists:", "model_dataset" in globals())

if "model_dataset" in globals():
    print("Rows:", len(model_dataset))
    print("Opportunity score exists:",
          "opportunity_score" in model_dataset.columns)

Training: 2025-10-01 to 2026-03-31
Validation: 2026-04-01 to 2026-06-30
model_dataset exists: True
Rows: 104173
Opportunity score exists: True


In [15]:
# Check that the historical opportunity score is ready for evaluation

print("Rows:", len(model_dataset))
print("Opportunity score available:",
      "opportunity_score" in model_dataset.columns)

print("\nOpportunity score statistics:")
print(model_dataset["opportunity_score"].describe())

Rows: 104173
Opportunity score available: True

Opportunity score statistics:
count    104173.000000
mean          0.012219
std           0.011333
min           0.000000
25%           0.000000
50%           0.010094
75%           0.020595
max           0.069787
Name: opportunity_score, dtype: float64


In [16]:
top_opportunities = (
    model_dataset
    .sort_values("opportunity_score", ascending=False)
    [[
        "client_hash_id",
        "content_hash_id",
        "position_bucket",
        "ctr",
        "baseline_ctr",
        "historical_ctr_gap",
        "impressions",
        "active_days",
        "opportunity_score"
    ]]
    .head(20)
)

top_opportunities

,client_hash_id,content_hash_id,position_bucket,ctr,baseline_ctr,historical_ctr_gap,impressions,active_days,opportunity_score
57129,client_73cda7b4e4f265ea,content_fec55986a1868d62,1-3,0.000003,0.005443,0.005440,372381.0,182,0.069787
5107,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1-3,0.000011,0.005443,0.005432,366197.0,182,0.069590
10565,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1-3,0.000007,0.005443,0.005436,279673.0,135,0.068173
87800,client_23a62021009f63c4,content_44f34c0a90047651,1-3,0.000132,0.005443,0.005311,302868.0,65,0.067030
77007,client_e547b89c05043229,content_306bc78dff1eb683,1-3,0.000504,0.005443,0.004939,242280.0,135,0.061239
33084,client_23a62021009f63c4,content_2ac8c7995de53cd1,1-3,0.000191,0.005443,0.005252,94198.0,78,0.060151
77010,client_e547b89c05043229,content_c46df0fa61530d86,1-3,0.000487,0.005443,0.004956,170379.0,135,0.059697
23825,client_e547b89c05043229,content_35f10273f0baf406,1-3,0.000214,0.005443,0.005229,79593.0,135,0.059012
47681,client_23a62021009f63c4,content_bf078007df823490,1-3,0.000000,0.005443,0.005443,44707.0,20,0.058283
8418,client_23a62021009f63c4,content_66d3e7a515e4ec68,1-3,0.000132,0.005443,0.005311,53174.0,171,0.057795


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
